# Classification NBA Model

## Configuration

## Imports

In [ ]:
import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    make_scorer,
    precision_score,
    recall_score,
)
from sklearn.model_selection import KFold, cross_validate, train_test_split
try:
    from utils import clean_dataframe_for_training
except ImportError:
    from nba_ou.data_processing.missing_data.clean_df_for_training import (
        clean_dataframe_for_training
    )

## Load Data

In [ ]:
data_path = "/home/adrian_alvarez/Projects/NBA_over_under_predictor/data/train_data/"
name = "all_odds_training_data_until_20260120.csv"

path = data_path + name

# path = "/home/coder/repos/terminus-eu-complian/data/energy_labels_tabular.csv"

df_stats = pd.read_csv(path)

dtype_dict = {col: str for col in df_stats.columns if "ID" in col.upper()}

df_stats = pd.read_csv(
    path,
    dtype=dtype_dict
)
df_stats['GAME_DATE'] = pd.to_datetime(df_stats['GAME_DATE']).dt.strftime('%Y-%m-%d')

In [ ]:
# #print column names with name in lower case
# for col in df_stats.columns:
#     if not col.isupper():
#         print(col)

In [ ]:
#filter for last two seasons, gett seasons 2025 and 2024
# df_stats = df_stats[df_stats['SEASON_YEAR'].isin([2024, 2025])]

In [ ]:
df_stats

In [ ]:
df_to_train = clean_dataframe_for_training(df_stats, nan_threshold=3, drop_all_na_rows=True,keep_columns =['GAME_DATE', 'TEAM_NAME_TEAM_HOME', 'TEAM_NAME_TEAM_AWAY'], verbose=1)


In [ ]:
# Count NAs per column
na_counts = df_to_train.isna().sum()

# Get most common SEASON_YEAR for nulls in each column
most_common_season = []
for col in df_to_train.columns:
    if na_counts[col] > 0:
        # Get rows where this column is null
        null_rows = df_stats[df_stats[col].isna()]
        if len(null_rows) > 0 and 'SEASON_YEAR' in df_stats.columns:
            # Find most common SEASON_YEAR for these null rows
            common_season = null_rows['SEASON_YEAR'].mode()
            most_common_season.append(common_season.iloc[0] if len(common_season) > 0 else None)
        else:
            most_common_season.append(None)
    else:
        most_common_season.append(None)

na_counts_df = pd.DataFrame({
    'Column': na_counts.index,
    'NA_Count': na_counts.values,
    'NA_Percentage': (na_counts.values / len(df_to_train) * 100).round(2),
    'Most_Common_Season_Year': most_common_season
}).sort_values('NA_Count', ascending=False)

# Show only columns with NAs
na_counts_df[na_counts_df['NA_Count'] > 0]

In [ ]:
df_to_train['LINE_ERROR'] = df_to_train['TOTAL_POINTS'] - df_to_train['TOTAL_OVER_UNDER_LINE']

In [ ]:
# df_to_train = df_to_train[df_to_train['LINE_ERROR'] != 0]

## Train / Test

In [ ]:
import numpy as np
import pandas as pd

from tabpfn_time_series import TabPFNTSPipeline, TabPFNMode
from tabpfn_time_series.plot import plot_forecast

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


def build_ts_frames(
    df: pd.DataFrame,
    timestamp_col: str = "GAME_DATE",
    target_col: str = "LINE_ERROR",
    col_item_id: str = None,
    drop_leakage_cols: tuple[str, ...] = ("TOTAL_POINTS",),
) -> pd.DataFrame:
    """
    Produce a TabPFN TS-compatible long dataframe:
    columns: item_id, timestamp, target, + covariates
    """
    out = df.copy()

    # timestamp must be datetime, not string
    out[timestamp_col] = pd.to_datetime(out[timestamp_col], errors="coerce")
    out = out.dropna(subset=[timestamp_col])

    if col_item_id is not None:
        out["item_id"] = out[col_item_id]
    else:
        out["item_id"] = "NBA"

    out["timestamp"] = out[timestamp_col]
    out["target"] = pd.to_numeric(out[target_col], errors="coerce")

    # Drop obvious leakage columns and non covariate columns
    to_drop = {timestamp_col, target_col,col_item_id, "SEASON_YEAR", "item_id", "timestamp", "target"}
    to_drop |= set(drop_leakage_cols)

    covariate_cols = [c for c in out.columns if c not in to_drop]

    tsdf = out[["item_id", "timestamp", "target"] + covariate_cols].sort_values(
        ["item_id", "timestamp"]
    )

    # Remove rows with missing target in the historical context
    tsdf = tsdf.dropna(subset=["target"])

    return tsdf



In [ ]:
import numpy as np
import pandas as pd

HOME_SUFFIX = "_TEAM_HOME"
AWAY_SUFFIX = "_TEAM_AWAY"

def make_team_panel_from_games(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["GAME_DATE"] = pd.to_datetime(df["GAME_DATE"], errors="coerce")
    df = df.dropna(subset=["GAME_DATE"])

    # Target at game level
    df["LINE_ERROR"] = df["TOTAL_POINTS"] - df["TOTAL_OVER_UNDER_LINE"]

    home = df.copy()
    away = df.copy()

    home["item_id"] = home["TEAM_NAME_TEAM_HOME"]
    away["item_id"] = away["TEAM_NAME_TEAM_AWAY"]

    home["IS_HOME"] = 1
    away["IS_HOME"] = 0

    home["OPPONENT"] = home["TEAM_NAME_TEAM_AWAY"]
    away["OPPONENT"] = away["TEAM_NAME_TEAM_HOME"]

    # Unify *_TEAM_HOME and *_TEAM_AWAY -> *_TEAM
    home_cols = [c for c in df.columns if c.endswith(HOME_SUFFIX)]
    for c_home in home_cols:
        base = c_home[: -len(HOME_SUFFIX)]
        c_away = base + AWAY_SUFFIX
        if c_away in df.columns:
            home[base + "_TEAM"] = home[c_home]
            away[base + "_TEAM"] = away[c_away]

    # Drop original paired columns to reduce dimensionality
    drop_pair_cols = []
    for c_home in home_cols:
        base = c_home[: -len(HOME_SUFFIX)]
        c_away = base + AWAY_SUFFIX
        if c_away in df.columns:
            drop_pair_cols.extend([c_home, c_away])

    home = home.drop(columns=drop_pair_cols, errors="ignore")
    away = away.drop(columns=drop_pair_cols, errors="ignore")

    panel = pd.concat([home, away], ignore_index=True)

    # Keep only what you need; do NOT keep TOTAL_POINTS in covariates
    panel = panel.drop(columns=["TOTAL_POINTS"], errors="ignore")

    # Rename for TabPFN TS
    panel = panel.rename(columns={"GAME_DATE": "timestamp", "LINE_ERROR": "target"})

    # Final ordering
    panel = panel.sort_values(["item_id", "timestamp"]).dropna(subset=["target"])

    return panel


In [ ]:
tsdf = make_team_panel_from_games(df_to_train)

N_CONTEXT = 15  # tune this

tmp = (
    tsdf.groupby("item_id", group_keys=False)
        .tail(N_CONTEXT + 1)
        .copy()
)

test_df = tmp.groupby("item_id", group_keys=False).tail(1).copy()
context_df = tmp.drop(index=test_df.index).copy()
future_df = test_df.drop(columns=["target"]).copy()


In [ ]:
# Run TabPFN TS
pipeline = TabPFNTSPipeline(tabpfn_mode=TabPFNMode.CLIENT, tabpfn_model_config={"ignore_pretraining_limits": True},
)  # or TabPFNMode.CLIENT
pred_df = pipeline.predict_df(
    context_df=context_df,
    future_df=future_df,
    # If your version supports custom quantiles you can pass them, otherwise omit.
    # quantiles=[0.1, 0.5, 0.9],
)


In [ ]:

# Merge predictions with ground truth
pred_reset = pred_df.reset_index()  # expects item_id,timestamp index in the returned df
eval_df = test_df.merge(pred_reset, on=["item_id", "timestamp"], how="left", validate="one_to_one")

# Pick the median forecast (handle both float and string column naming)
q50_col = 0.6 if 0.6 in eval_df.columns else "0.6"
y_true = eval_df["target_x"].to_numpy(dtype=float)
y_pred = eval_df[q50_col].to_numpy(dtype=float)

print("MSE:", mean_squared_error(y_true, y_pred))
print("MAE:", mean_absolute_error(y_true, y_pred))
print("R2 :", r2_score(y_true, y_pred))

# Optional: visualize
some_teams = context_df["item_id"].unique()[:3].tolist()
plot_forecast(context_df, pred_df, test_df=test_df, item_ids=some_teams)

In [ ]:
pred_reset['real_target'] = y_true

In [ ]:
pred_reset['EQUAL_SIGN'] = np.sign(pred_reset[0.3]) == np.sign(pred_reset['real_target'])    

In [ ]:
pred_reset['EQUAL_SIGN'].mean()

In [ ]:
pred_reset

In [ ]:
def error_sign_accuracy(y_true_error, y_pred_error, include_pushes: bool = False) -> float:
    y_true_error = np.asarray(y_true_error, dtype=float)
    y_pred_error = np.asarray(y_pred_error, dtype=float)

    true_sign = np.sign(y_true_error)
    pred_sign = np.sign(y_pred_error)

    if include_pushes:
        valid = ~((true_sign == 0) ^ (pred_sign == 0))
    else:
        valid = (true_sign != 0) & (pred_sign != 0)

    if not np.any(valid):
        return np.nan

    return float(np.mean(true_sign[valid] == pred_sign[valid]))

print("Directional accuracy:", error_sign_accuracy(y_true, y_pred))


In [ ]:
acc = error_sign_accuracy(future_df_target['target'], pred_df['target'])
acc

In [ ]:
future_df_target['target'], pred_df['target']

In [ ]:
pred_df

In [ ]:
future_df_target['target']